In [ ]:
import os
import cv2
import json
import torch
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import tensorflow.keras.backend as K
from pyspark.sql import SparkSession
from glob import glob as gg
from zipfile import ZipFile
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, BatchNormalization,
    Permute, Reshape, LSTM, Dense, Dropout, Lambda
)
from pyspark.sql.functions import col, aggregate as agg, max, min, filter
from sklearn.model_selection import train_test_split
# from tensorflow.keras.metrics import Precision, Recall, F1Score
from tensorflow.keras.models import load_model, Model, Sequential
from pyspark.sql.types import StringType, ArrayType, IntegerType, FloatType

In [3]:
spark = SparkSession.builder \
            .appName("LSTMJsonExtractor") \
            .config("spark.driver.port", "5432") \
            .config("spark.blockManager.port", "4321") \
            .config("spark.driver.memory", "8g") \
            .config("spark.executor.memory", "8g") \
            .config("spark.driver.maxResultSize", "8g") \
            .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/16 08:36:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/04/16 08:36:47 WARN Utils: Service 'sparkDriver' could not bind on port 5432. Attempting port 5433.
25/04/16 08:36:47 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/04/16 08:36:48 WARN Utils: Service 'org.apache.spark.network.netty.NettyBlockTransferService' could not bind on port 4321. Attempting port 4322.


In [4]:
images_dir = "archive/batch_*/background_images/*.jpg"

image_paths = gg(f"{images_dir}")

json_dir = "archive/batch_*/json/*"

json_paths = gg(f"{json_dir}")

lookup_dict = {os.path.basename(path): path for path in image_paths}

broadcast_lookup = spark.sparkContext.broadcast(lookup_dict)

In [ ]:
with open("symbols_dict.json", "r") as file:
    symbols_index = json.load(file)

In [ ]:
def locate_image_path(row):
    
    image_filename = row.asDict().get("filename", None)

    if image_filename:
        return broadcast_lookup.value.get(image_filename, None)
    return None

def process_lstm_input(row):
    row_dict = row.asDict()
    image_path = locate_image_path(row)
    
    if image_path:

        image_data = cv2.imread(image_path)
        image_data = cv2.resize(image_data, 500, 1500)
        image_data = image_data.astype(np.float32) / 255.0

        label_indexes = [symbols_index.get(char) for char in row_dict["visible_latex_chars"]]
        
        label_len = len(label_indexes)

        return [
            image_path, 
            image_data,
            row_dict["latex"], 
            # row_dict["full_latex_chars"], 
            row_dict["visible_latex_chars"], 
            len(row_dict["visible_latex_chars"]), 
        ]
        
    return None
        
def ctc_lambda_func(args):
    y_pred, labels, input_length, label_length = args
    return K.ctc_batch_cost(labels, y_pred, input_length, label_length)

In [6]:
temp = spark.read.parquet("data/images_small_dims.parquet")

temp.show(5)

+--------------------+--------------------+--------------------+-----+------+-----+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|            filename|               latex| visible_latex_chars|width|height|depth|               xmins|               xmaxs|               ymins|               ymaxs|           xmins_raw|           xmaxs_raw|           ymins_raw|           ymaxs_raw|           png_masks|
+--------------------+--------------------+--------------------+-----+------+-----+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|436008a6-638f-4bd...|\lim_{x\to1^{+}}-...|[\lim_, x, \to, 1...|941.0| 265.0|  4.0|[0.023404256, 0.0...|[0.25851065, 0.08...|[0.053030305, 0.8...|[0.594697, 0.9810...|[22, 55, 91, 1

In [39]:
data = []

for row in temp.collect():
    result = process_lstm_input(row)
    
    if result is not None:
        data.append(result)

# temp.rdd.toLocalIterator(process_lstm_input)

In [14]:
import tensorflow as tf

img_height = 64
img_width = 256
n_channels = 3
n_classes = 100


input_img = Input(shape=(img_height, img_width, n_channels), name='image_input')
labels = Input(name='labels', shape=(None,), dtype='int32')
input_length = Input(name='input_length', shape=(1,), dtype='int64')
label_length = Input(name='label_length', shape=(1,), dtype='int64')


x = Conv2D(32, (3,3), activation='relu', padding='same')(input_img)
x = BatchNormalization()(x)
x = MaxPooling2D(pool_size=(2,2))(x)

x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = MaxPooling2D(pool_size=(2,2))(x)

x = Conv2D(128, (3,3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = MaxPooling2D(pool_size=(2,2))(x)

x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = MaxPooling2D(pool_size=(2,2))(x)

x = Permute((2,1,3))(x)

new_shape = (x.shape[1], x.shape[2] * x.shape[3])
x = Reshape(target_shape=new_shape)(x)

lstm_out = LSTM(128, return_sequences=True, activation='tanh')(x)
lstm_out = Dropout(0.2)(lstm_out)
lstm_out = LSTM(128, return_sequences=True, activation='tanh')(lstm_out)

y_pred = Dense(n_classes, activation='softmax', name='y_pred')(lstm_out)

loss_out = Lambda(ctc_lambda_func, output_shape=(1,), name='ctc')([
                                                                    y_pred, 
                                                                    labels, 
                                                                    input_length, 
                                                                    label_length
                                                                ])

model = Model(inputs = [input_img, labels, input_length, label_length], outputs = loss_out)

model.compile(optimizer='adam', loss={'ctc': lambda y_true, y_pred: y_pred})

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 64, 256,   │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 64, 256,   │        896 │ image_input[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 64, 256,   │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 32, 128,   │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 32, 128,   │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 128,   │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 16, 64,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 16, 64,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 64,    │        512 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 8, 32,     │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 8, 32, 64) │     73,792 │ max_pooling2d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 8, 32, 64) │        256 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 4, 16, 64) │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ permute (Permute)   │ (None, 16, 4, 64) │          0 │ max_pooling2d_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 16, 256)   │          0 │ permute[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 16, 128)   │    197,120 │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 16, 128)   │          0 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 16, 128)   │    131,584 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ y_pred (Dense)      │ (None, 16, 100)   │     12,900 │ lstm_1[0][0]    

 Total params: 509,796 (1.94 MB)

 Trainable params: 509,220 (1.94 MB)

 Non-trainable params: 576 (2.25 KB)

In [ ]:
training = model.fit(
                    data, 
                    validation_data = data
                    batch_size = 64,
                    epochs = 50,
                    verbose = 1)